# Decline vs Signed & Bound ML Solution

This notebook builds a Python ML approach for the current scope:

- **Exclude NTU completely**
- Use only `Declined` and `Signed & Bound` records
- Train a model to estimate how closely each declined case resembles historically signed & bound cases
- Identify prominent variables associated with decline
- Create final output tables that can be written back to Snowflake

Recommended model approach in this notebook:

1. **Random Forest classifier** as the primary model because it handles non-linear patterns, mixed business/risk features, and gives feature importance.
2. **Logistic Regression** as an explainable benchmark.
3. Optional **XGBoost** section if the package is available in your environment.

Main outputs:

- `decline_opportunity_review.csv`
- `decline_driver_summary.csv`
- optional write-back to Snowflake


## 1. Install/import packages

Run installation cells only if your environment does not already have the packages. In company environments, package installation may be restricted.

In [ ]:
# Optional installs - uncomment if needed
# %pip install snowflake-connector-python pandas scikit-learn openpyxl joblib matplotlib
# %pip install xgboost  # optional, only if allowed


In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

import matplotlib.pyplot as plt
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 300)


## 2. Configuration

Update these values if your database, schema, or table name is different.

In [ ]:
DB_NAME = 'EXPERIMENT_TEAM_DB'
SCHEMA_NAME = 'PUBLIC'
SOURCE_TABLE = 'ML_DATA_PHASE1'

TARGET_STATUS_COL = 'UWRISK_INWARDINSUREDLINE_LINESTATUS'
SUBMISSION_REF_COL = 'UWRISK_INWARDSUBMISSION_CONVEXSUBMISSIONREFERENCE'

SIGNED_LABEL = 'Signed & Bound'
DECLINED_LABEL = 'Declined'
TARGET_COL = 'SIGNED_BOUND_TARGET'

OUTPUT_DIR = Path('./decline_ml_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)


## 3. Load data from Snowflake

This cell uses environment variables for Snowflake login. Set these before running the notebook, or replace them directly in the connection block.

Expected environment variables:

- `SNOWFLAKE_ACCOUNT`
- `SNOWFLAKE_USER`
- `SNOWFLAKE_PASSWORD`
- `SNOWFLAKE_WAREHOUSE`
- `SNOWFLAKE_ROLE`

If direct Python Snowflake access is not allowed, export `ML_DATA_PHASE1` from Snowflake to CSV and use the CSV fallback cell below.

In [ ]:
USE_SNOWFLAKE = True  # set False if using CSV fallback

if USE_SNOWFLAKE:
    import snowflake.connector

    conn = snowflake.connector.connect(
        account=os.getenv('SNOWFLAKE_ACCOUNT'),
        user=os.getenv('SNOWFLAKE_USER'),
        password=os.getenv('SNOWFLAKE_PASSWORD'),
        warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
        database=DB_NAME,
        schema=SCHEMA_NAME,
        role=os.getenv('SNOWFLAKE_ROLE'),
    )

    sql = f'''
    SELECT *
    FROM {DB_NAME}.{SCHEMA_NAME}.{SOURCE_TABLE}
    WHERE {TARGET_STATUS_COL} IN ('{DECLINED_LABEL}', '{SIGNED_LABEL}')
    '''

    df = pd.read_sql(sql, conn)
    print(df.shape)
    display(df.head())
else:
    print('Using CSV fallback. Run the next cell instead.')


In [ ]:
# CSV fallback - use this if you export ML_DATA_PHASE1 to CSV from Snowflake
# USE_SNOWFLAKE = False
# csv_path = 'ML_DATA_PHASE1.csv'
# df = pd.read_csv(csv_path)
# df = df[df[TARGET_STATUS_COL].isin([DECLINED_LABEL, SIGNED_LABEL])].copy()
# print(df.shape)
# display(df.head())


## 4. Create target and define feature groups

The target is:

- `1` = Signed & Bound
- `0` = Declined

We exclude identifiers and the line status from model training.

In [ ]:
df.columns = [c.upper() for c in df.columns]
TARGET_STATUS_COL = TARGET_STATUS_COL.upper()
SUBMISSION_REF_COL = SUBMISSION_REF_COL.upper()
TARGET_COL = TARGET_COL.upper()

# Create target
_df = df.copy()
_df[TARGET_COL] = np.where(_df[TARGET_STATUS_COL] == SIGNED_LABEL, 1, 0)

print(_df[TARGET_STATUS_COL].value_counts(dropna=False))
print('
Target distribution:')
print(_df[TARGET_COL].value_counts(normalize=True).rename('percentage'))


In [ ]:
# Columns based on your ML_DATA_PHASE1 screenshots.
# The notebook will automatically keep only columns that exist in your dataframe.

categorical_features = [
    'UWRISK_INWARDRISKSECTION_PLACINGBASIS',
    'UWRISK_INWARDSUBMISSION_ACTIVITY1SEGMENT',
    'UWRISK_INWARDSUBMISSION_NEWORRENEWAL',
    'UWRISK_INWARDSUBMISSION_BROKERNAME',
    'UWRISK_INWARDRISKSECTION_CLASSOFBUSINESSTIER2NAME',
    'UWRISK_INWARDRISKSECTION_TARGETMARKET',
    'OED_UWRISK_PREDOMINANTEXPOSUREREGION',
    'OED_SINGLESITEDOMINANCECATEGORY',
    'EXPOSURE_REGION_GROUP',
    'TOTALTIV_BAND',
    'OED_DOMINANT_CONSTRUCTION_CLASS',
    'OED_DOMINANT_ISO_CONSTRUCTION_CODE',
    'OED_DOMINANT_OCCUPANCY_CLASS',
    'OED_DOMINANT_ROOF_ANCHORAGE',
    'OED_DOMINANT_ROOF_ANCHORAGE_RISK_CATEGORY',
    'EXT_FL_HIGHESTRISKCATEGORY',
    'EXT_WS_HIGHESTRISKCATEGORY',
    'EXT_EQ_HIGHESTRISKCATEGORY',
    'LAST5YEARS_NUMBEROFCLAIMSBAND',
    'LAST10YEARS_NUMBEROFCLAIMSBAND',
    'PORTFOLIO_BURNING_COST_LAST5YEARS_BANDS',
]

numeric_features = [
    'UWRISK_FAVOCCUPANCYFLAG',
    'UWRISK_HEAVYOCCUPANCYFLAG',
    'UWRISK_POORAPPRAISALVALUESFLAG',
    'UWRISK_BROKERCHURNFLAG',
    'UWRISK_FLUCTUATINGTIVFLAG',
    'OED_LOCATIONSCOUNT',
    'OED_SUBMISSIONCONCENTRATIONRATIO',
    'OED_TIV_CONTENTS_PERC',
    'OED_DOMINANT_LOCATION_TIV_SHARE',
    'OED_EXPO_CONSTRUCTION_CODE_HIGHRISK',
    'OED_EXPO_CONSTRUCTION_CODE_MEDIUMRISK',
    'OED_EXPO_CONSTRUCTION_CODE_SPECIALIZEDRISK',
    'OED_DOMINANT_YEAR_BUILT',
    'OED_EXPO_YEARBUILT_PRE1975',
    'OED_EXPO_YEARBUILT_1975_2000',
    'OED_EXPO_YEARBUILT_POST2000',
    'OED_DOMINANT_NUMSTORIES',
    'OED_EXPO_NUMSTORIES_1_3',
    'OED_EXPO_NUMSTORIES_4_7',
    'OED_EXPO_NUMSTORIES_8PLUS',
    'OED_EXPO_SQFOOTAGE_LESSTHAN25K',
    'OED_EXPO_SQFOOTAGE_25K_100K',
    'OED_EXPO_SQFOOTAGE_MORETHAN100K',
    'OED_EXPO_OCCUPANCY_HIGHRISK',
    'OED_EXPO_OCCUPANCY_MEDIUMRISK',
    'OED_EXPO_OCCUPANCY_LOWRISK',
    'OED_EXPO_BASEMENT_LOCS',
    'OED_EXPO_MORETHANONEBUILDING_LOCS',
    'EXT_EXPO_HIGHRISKFLOOD',
    'EXT_EXPO_HIGHRISKWS',
    'EXT_WS_HIGHESTRISKGROUPFLAG',
    'EXT_WS_AVERAGERISKSCORE',
    'EXT_EXPO_HIGHRISKEQ',
    'OED_EXPO_PROTECTION_NOSPRINKLRPERC',
    'OED_PROTECTION_DOMINANTLOC_NOSPRINKLR',
    'AI_PROTECTION_SPRINKLERYES_PERC',
    'AI_PROTECTION_SPRIKLERTYPE_PREACTIONFLAG',
    'AI_PROTECTION_SPRIKLERTYPE_DELUGEFLAG',
    'AI_PROTECTION_FIREALARMTYPE_ANYCENTRALSTATIONFLAG',
    'AI_PROTECTION_FIREALARMTYPE_ANYLOCALALARMFLAG',
    'AI_PROTECTION_PPCODE_ANYPREFERREDFLAG',
    'AI_PROTECTION_PPCODE_ANYSTANDARDFLAG',
    'AI_PROTECTION_PPCODE_ANYRESTRICTEDFLAG',
    'AI_PROTECTION_PPCODE_PREFERREDPERC',
    'AI_PROTECTION_EQRESISTIVESYSTEM_ANYFLAG',
    'AI_PROTECTION_ROOFMATERIAL_ANYHIGHPROTECTIONFLAG',
    'AI_PROTECTION_ROOFMATERIAL_ANYLOWPROTECTIONFLAG',
    'AI_PROTECTION_ROOFMATERIAL_HIGHPROTECTIONPERC',
    'AI_PROTECTION_ROOFANCHORAGE_ANYHIGHPROTECTIONFLAG',
    'AI_PROTECTION_ROOFANCHORAGE_ANYLOWPROTECTIONFLAG',
    'AI_PROTECTION_ROOFANCHORAGE_HIGHPROTECTIONPERC',
    'DERIVED_WOODFRAME_RESTRICTEDPPC_FLAG',
    'DERIVED_WOODFRAME_NOSPRINKLER_FLAG',
    'LOSSFREQUENCY',
]

categorical_features = [c for c in categorical_features if c in _df.columns]
numeric_features = [c for c in numeric_features if c in _df.columns]
feature_cols = categorical_features + numeric_features

print(f'Categorical features: {len(categorical_features)}')
print(f'Numeric features: {len(numeric_features)}')
print(f'Total features: {len(feature_cols)}')


## 5. Basic data quality checks

This helps identify columns with high missingness or low variation.

In [ ]:
missing_summary = (
    _df[feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_summary.columns = ['FEATURE', 'MISSING_RATE']
display(missing_summary.head(30))

low_variation = []
for c in feature_cols:
    nunique = _df[c].nunique(dropna=True)
    low_variation.append((c, nunique))
low_variation = pd.DataFrame(low_variation, columns=['FEATURE', 'N_UNIQUE']).sort_values('N_UNIQUE')
display(low_variation.head(30))


## 6. Train/test split

We use stratified split so both classes are represented in train and test.

In [ ]:
X = _df[feature_cols].copy()
y = _df[TARGET_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print('Train:', X_train.shape, y_train.value_counts().to_dict())
print('Test :', X_test.shape, y_test.value_counts().to_dict())


## 7. Preprocessing pipeline

Categorical features are one-hot encoded. Numeric features are median-imputed and scaled for logistic regression. Random Forest does not require scaling, but using one consistent preprocessing pipeline is practical.

In [ ]:
try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=True)

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', ohe)
])

numeric_transformer_scaled = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler(with_mean=False))
])

numeric_transformer_tree = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

preprocess_scaled = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', numeric_transformer_scaled, numeric_features),
    ],
    remainder='drop'
)

preprocess_tree = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', numeric_transformer_tree, numeric_features),
    ],
    remainder='drop'
)


## 8. Model 1: Logistic Regression baseline

This gives an explainable benchmark. It may underperform if relationships are non-linear.

In [ ]:
logistic_model = Pipeline(steps=[
    ('preprocess', preprocess_scaled),
    ('model', LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        solver='saga',
        n_jobs=-1,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)
log_pred = logistic_model.predict(X_test)
log_prob = logistic_model.predict_proba(X_test)[:, 1]

log_metrics = {
    'model': 'Logistic Regression',
    'accuracy': accuracy_score(y_test, log_pred),
    'precision': precision_score(y_test, log_pred, zero_division=0),
    'recall': recall_score(y_test, log_pred, zero_division=0),
    'f1': f1_score(y_test, log_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, log_prob) if y_test.nunique() == 2 else np.nan,
}
print(log_metrics)
print('
Classification report:')
print(classification_report(y_test, log_pred, target_names=['Declined', 'Signed & Bound']))
print('Confusion matrix:')
print(confusion_matrix(y_test, log_pred))


## 9. Model 2: Random Forest primary model

This is the recommended model for the first Python version because it is robust, handles mixed feature types after encoding, captures non-linear patterns, and gives feature importance.

In [ ]:
rf_model = Pipeline(steps=[
    ('preprocess', preprocess_tree),
    ('model', RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=5,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

rf_metrics = {
    'model': 'Random Forest',
    'accuracy': accuracy_score(y_test, rf_pred),
    'precision': precision_score(y_test, rf_pred, zero_division=0),
    'recall': recall_score(y_test, rf_pred, zero_division=0),
    'f1': f1_score(y_test, rf_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, rf_prob) if y_test.nunique() == 2 else np.nan,
}
print(rf_metrics)
print('
Classification report:')
print(classification_report(y_test, rf_pred, target_names=['Declined', 'Signed & Bound']))
print('Confusion matrix:')
print(confusion_matrix(y_test, rf_pred))


## 10. Compare models

In [ ]:
model_metrics = pd.DataFrame([log_metrics, rf_metrics]).sort_values('roc_auc', ascending=False)
display(model_metrics)

best_model = rf_model if rf_metrics['roc_auc'] >= log_metrics['roc_auc'] else logistic_model
best_model_name = 'Random Forest' if best_model is rf_model else 'Logistic Regression'
print('Selected model:', best_model_name)


## 11. ROC curve

In [ ]:
plt.figure(figsize=(7, 5))
for name, prob in [('Logistic Regression', log_prob), ('Random Forest', rf_prob)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} AUC={auc:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


## 12. Feature importance aggregated to original variables

Random Forest feature importance is calculated at encoded-feature level. This section aggregates one-hot encoded columns back to original feature names, which is easier for business users.

In [ ]:
def get_feature_names_from_column_transformer(ct):
    names = []
    for transformer_name, transformer, cols in ct.transformers_:
        if transformer_name == 'remainder' and transformer == 'drop':
            continue
        if transformer_name == 'cat':
            ohe_step = transformer.named_steps['onehot']
            try:
                cat_names = list(ohe_step.get_feature_names_out(cols))
            except AttributeError:
                cat_names = list(ohe_step.get_feature_names(cols))
            names.extend(cat_names)
        elif transformer_name == 'num':
            names.extend(cols)
    return names

# Use RF for feature importance even if logistic slightly wins, because RF importance is easier to aggregate.
rf_preprocess = rf_model.named_steps['preprocess']
rf_estimator = rf_model.named_steps['model']
encoded_feature_names = get_feature_names_from_column_transformer(rf_preprocess)
importances = rf_estimator.feature_importances_

encoded_importance = pd.DataFrame({
    'encoded_feature': encoded_feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

# Aggregate one-hot features back to original column names
original_importance_rows = []
for feature in feature_cols:
    if feature in numeric_features:
        mask = encoded_importance['encoded_feature'] == feature
    else:
        mask = encoded_importance['encoded_feature'].str.startswith(feature + '_')
    original_importance_rows.append({
        'feature_name': feature,
        'importance': encoded_importance.loc[mask, 'importance'].sum()
    })

feature_importance = pd.DataFrame(original_importance_rows).sort_values('importance', ascending=False)
feature_importance['importance_rank'] = range(1, len(feature_importance) + 1)
display(feature_importance.head(30))

feature_importance.to_csv(OUTPUT_DIR / 'model_feature_importance.csv', index=False)


## 13. Decline driver summary using direct rate analysis

This is critical because feature importance tells importance, while decline-rate analysis tells business direction.

In [ ]:
def driver_summary_for_feature(data, feature, min_count=5):
    temp = data[[TARGET_STATUS_COL, TARGET_COL, feature]].copy()
    temp = temp[temp[feature].notna()]
    temp[feature] = temp[feature].astype(str)
    summary = (
        temp.groupby(feature)
        .agg(
            total_records=(TARGET_COL, 'size'),
            signed_bound_count=(TARGET_COL, 'sum')
        )
        .reset_index()
        .rename(columns={feature: 'feature_value'})
    )
    summary['declined_count'] = summary['total_records'] - summary['signed_bound_count']
    summary['decline_rate_percent'] = 100 * summary['declined_count'] / summary['total_records']
    summary['signed_bound_rate_percent'] = 100 * summary['signed_bound_count'] / summary['total_records']
    summary['feature_name'] = feature
    summary = summary[summary['total_records'] >= min_count]
    return summary[['feature_name', 'feature_value', 'total_records', 'declined_count', 'signed_bound_count', 'decline_rate_percent', 'signed_bound_rate_percent']]

summaries = []
for f in feature_cols:
    try:
        summaries.append(driver_summary_for_feature(_df, f, min_count=5))
    except Exception as e:
        print('Skipped', f, e)

decline_driver_summary = pd.concat(summaries, ignore_index=True)

# Add model importance rank if available
importance_lookup = feature_importance[['feature_name', 'importance', 'importance_rank']]
decline_driver_summary = decline_driver_summary.merge(importance_lookup, on='feature_name', how='left')
decline_driver_summary = decline_driver_summary.sort_values(['decline_rate_percent', 'total_records'], ascending=[False, False])

display(decline_driver_summary.head(50))
decline_driver_summary.to_csv(OUTPUT_DIR / 'decline_driver_summary.csv', index=False)


## 14. Score all declined cases

The model score is interpreted as:

- Higher `SIGNED_BOUND_PROBABILITY` = declined case looks more similar to historically signed-bound cases
- Lower score = declined case looks more similar to historically declined cases

Then we add a simple business overlay to create `FINAL_OPPORTUNITY_SCORE`. You can tune these adjustments with underwriters.

In [ ]:
# Score every record, then filter declined
all_prob = best_model.predict_proba(_df[feature_cols])[:, 1]
_df['SIGNED_BOUND_PROBABILITY'] = all_prob

review = _df[_df[TARGET_STATUS_COL] == DECLINED_LABEL].copy()

# Business overlay. Adjust these values after business validation.
review['BUSINESS_ADJUSTMENT'] = 0.0

if 'UWRISK_FAVOCCUPANCYFLAG' in review.columns:
    review['BUSINESS_ADJUSTMENT'] += np.where(review['UWRISK_FAVOCCUPANCYFLAG'].fillna(0).astype(float) == 1, 0.05, 0)
if 'UWRISK_INWARDSUBMISSION_NEWORRENEWAL' in review.columns:
    review['BUSINESS_ADJUSTMENT'] += np.where(review['UWRISK_INWARDSUBMISSION_NEWORRENEWAL'].astype(str).str.lower() == 'renewal', 0.05, 0)
if 'UWRISK_POORAPPRAISALVALUESFLAG' in review.columns:
    review['BUSINESS_ADJUSTMENT'] -= np.where(review['UWRISK_POORAPPRAISALVALUESFLAG'].fillna(0).astype(float) == 1, 0.15, 0)
if 'UWRISK_HEAVYOCCUPANCYFLAG' in review.columns:
    review['BUSINESS_ADJUSTMENT'] -= np.where(review['UWRISK_HEAVYOCCUPANCYFLAG'].fillna(0).astype(float) == 1, 0.10, 0)
if 'UWRISK_BROKERCHURNFLAG' in review.columns:
    review['BUSINESS_ADJUSTMENT'] -= np.where(review['UWRISK_BROKERCHURNFLAG'].fillna(0).astype(float) == 1, 0.05, 0)
if 'UWRISK_FLUCTUATINGTIVFLAG' in review.columns:
    review['BUSINESS_ADJUSTMENT'] -= np.where(review['UWRISK_FLUCTUATINGTIVFLAG'].fillna(0).astype(float) == 1, 0.05, 0)
for hazard_col in ['EXT_FL_HIGHESTRISKCATEGORY', 'EXT_WS_HIGHESTRISKCATEGORY', 'EXT_EQ_HIGHESTRISKCATEGORY']:
    if hazard_col in review.columns:
        review['BUSINESS_ADJUSTMENT'] -= np.where(review[hazard_col].astype(str).str.lower().eq('high risk category'), 0.05, 0)

review['FINAL_OPPORTUNITY_SCORE'] = (review['SIGNED_BOUND_PROBABILITY'] + review['BUSINESS_ADJUSTMENT']).clip(0, 1)
review['REVIEW_PRIORITY'] = pd.cut(
    review['FINAL_OPPORTUNITY_SCORE'],
    bins=[-0.01, 0.40, 0.70, 1.01],
    labels=['Low Review Priority', 'Medium Review Priority', 'High Review Priority']
)


## 15. Build evidence summary for each declined record

In [ ]:
def build_evidence(row):
    evidence = []
    if row.get('SIGNED_BOUND_PROBABILITY', 0) >= 0.70:
        evidence.append('Model: high similarity to signed-bound cases')
    elif row.get('SIGNED_BOUND_PROBABILITY', 0) >= 0.40:
        evidence.append('Model: medium similarity to signed-bound cases')
    else:
        evidence.append('Model: low similarity to signed-bound cases')

    checks = [
        ('UWRISK_FAVOCCUPANCYFLAG', 1, 'Positive: favourable occupancy flag'),
        ('UWRISK_POORAPPRAISALVALUESFLAG', 1, 'Negative: poor appraisal values flag'),
        ('UWRISK_HEAVYOCCUPANCYFLAG', 1, 'Negative: heavy occupancy flag'),
        ('UWRISK_BROKERCHURNFLAG', 1, 'Negative: broker churn flag'),
        ('UWRISK_FLUCTUATINGTIVFLAG', 1, 'Negative: fluctuating TIV flag'),
    ]
    for col, val, msg in checks:
        if col in row.index:
            try:
                if pd.notna(row[col]) and float(row[col]) == val:
                    evidence.append(msg)
            except Exception:
                pass

    for col, msg in [
        ('EXT_FL_HIGHESTRISKCATEGORY', 'Negative: high flood risk'),
        ('EXT_WS_HIGHESTRISKCATEGORY', 'Negative: high windstorm risk'),
        ('EXT_EQ_HIGHESTRISKCATEGORY', 'Negative: high earthquake risk'),
    ]:
        if col in row.index and str(row[col]).lower() == 'high risk category':
            evidence.append(msg)

    if 'UWRISK_INWARDSUBMISSION_NEWORRENEWAL' in row.index and str(row['UWRISK_INWARDSUBMISSION_NEWORRENEWAL']).lower() == 'renewal':
        evidence.append('Positive: renewal business')

    return '; '.join(evidence)

review['EVIDENCE_SUMMARY'] = review.apply(build_evidence, axis=1)


## 16. Final declined opportunity output

In [ ]:
report_cols = [
    SUBMISSION_REF_COL,
    TARGET_STATUS_COL,
    'SIGNED_BOUND_PROBABILITY',
    'BUSINESS_ADJUSTMENT',
    'FINAL_OPPORTUNITY_SCORE',
    'REVIEW_PRIORITY',
    'EVIDENCE_SUMMARY',
    'UWRISK_INWARDSUBMISSION_BROKERNAME',
    'UWRISK_INWARDSUBMISSION_NEWORRENEWAL',
    'UWRISK_INWARDRISKSECTION_TARGETMARKET',
    'OED_UWRISK_PREDOMINANTEXPOSUREREGION',
    'EXPOSURE_REGION_GROUP',
    'TOTALTIV_BAND',
    'OED_DOMINANT_OCCUPANCY_CLASS',
    'OED_DOMINANT_CONSTRUCTION_CLASS',
]
report_cols = [c for c in report_cols if c in review.columns]

decline_opportunity_review = review[report_cols].sort_values('FINAL_OPPORTUNITY_SCORE', ascending=False)
display(decline_opportunity_review.head(50))

decline_opportunity_review.to_csv(OUTPUT_DIR / 'decline_opportunity_review.csv', index=False)


## 17. Optional: write outputs back to Snowflake

This requires permission to create/replace tables in your schema. If you do not have table creation permission, keep CSV outputs and upload them manually.

In [ ]:
# Optional write-back to Snowflake
# from snowflake.connector.pandas_tools import write_pandas
#
# if USE_SNOWFLAKE:
#     write_pandas(conn, decline_opportunity_review, 'PY_DECLINE_OPPORTUNITY_REVIEW', auto_create_table=True, overwrite=True)
#     write_pandas(conn, decline_driver_summary, 'PY_DECLINE_DRIVER_SUMMARY', auto_create_table=True, overwrite=True)
#     write_pandas(conn, feature_importance, 'PY_DECLINE_FEATURE_IMPORTANCE', auto_create_table=True, overwrite=True)
#     print('Written to Snowflake tables:')
#     print('PY_DECLINE_OPPORTUNITY_REVIEW')
#     print('PY_DECLINE_DRIVER_SUMMARY')
#     print('PY_DECLINE_FEATURE_IMPORTANCE')


## 18. Save model artifact locally

This lets you reuse the model without retraining, as long as the same columns are available.

In [ ]:
joblib.dump(best_model, OUTPUT_DIR / 'decline_signed_bound_best_model.joblib')
with open(OUTPUT_DIR / 'feature_columns.json', 'w') as f:
    json.dump({
        'categorical_features': categorical_features,
        'numeric_features': numeric_features,
        'feature_cols': feature_cols,
        'selected_model': best_model_name,
    }, f, indent=2)

print('Saved outputs in:', OUTPUT_DIR.resolve())
print('Files:')
for p in OUTPUT_DIR.iterdir():
    print('-', p.name)


## 19. How to explain the result

Use this interpretation for business:

- `SIGNED_BOUND_PROBABILITY` is the model's estimate of how similar a declined case is to historical signed-bound business.
- `FINAL_OPPORTUNITY_SCORE` applies a simple underwriting overlay to the model score.
- `REVIEW_PRIORITY` is not saying the decline was wrong; it identifies cases worth underwriting review.
- `decline_driver_summary.csv` shows which variable values have high decline rates and therefore may be prominent decline drivers.
